# Lesson 4: Building a Multi-Document Agent

## Setup

In [1]:
import nest_asyncio
nest_asyncio.apply()

## 1. Configure um agente em 3 papéis

**Nota**: Os arquivos pdf estão incluídos nesta lição. Para acessar esses documentos, vá para o menu `Arquivo` e selecione `Abrir...`.

In [3]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=hSyW5go0v8",
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "selfrag.pdf",
]

In [ ]:
! pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface

In [4]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, SummaryIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool, FunctionTool
from llama_index.core.vector_stores import MetadataFilters, FilterCondition
from typing import List, Optional

In [5]:
def get_doc_tools(
    file_path: str,
    name: str,
) -> str:
    """Get vector query and summary query tools from a document."""

    # load documents
    documents = SimpleDirectoryReader(input_files=[file_path]).load_data()
    splitter = SentenceSplitter(chunk_size=1024)
    nodes = splitter.get_nodes_from_documents(documents)
    vector_index = VectorStoreIndex(nodes)

    def vector_query(
        query: str,
        page_numbers: Optional[List[str]] = None
    ) -> str:
        """Use to answer questions over the MetaGPT paper.

        Useful if you have specific questions over the MetaGPT paper.
        Always leave page_numbers as None UNLESS there is a specific page you want to search for.

        Args:
            query (str): the string query to be embedded.
            page_numbers (Optional[List[str]]): Filter by set of pages. Leave as NONE
                if we want to perform a vector search
                over all pages. Otherwise, filter by the set of specified pages.

        """

        page_numbers = page_numbers or []
        metadata_dicts = [
            {"key": "page_label", "value": p} for p in page_numbers
        ]

        query_engine = vector_index.as_query_engine(
            similarity_top_k=2,
            filters=MetadataFilters.from_dicts(
                metadata_dicts,
                condition=FilterCondition.OR
            )
        )
        response = query_engine.query(query)
        return response


    vector_query_tool = FunctionTool.from_defaults(
        name=f"vector_tool_{name}",
        fn=vector_query
    )

    summary_index = SummaryIndex(nodes)
    summary_query_engine = summary_index.as_query_engine(
        response_mode="tree_summarize",
        use_async=True,
    )
    summary_tool = QueryEngineTool.from_defaults(
        name=f"summary_tool_{name}",
        query_engine=summary_query_engine,
        description=(
            "Use ONLY IF you want to get a holistic summary of MetaGPT. "
            "Do NOT use if you have specific questions over MetaGPT."
        ),
    )

    return vector_query_tool, summary_tool

In [7]:
import os

os.environ["ANTHROPIC_API_KEY"] = "YOUR_ANTHROPIC_API_KEY"
os.environ["HF_TOKEN"] = "YOUR_HUGGINGFACE_TOKEN"

llm = Anthropic(model="claude-3-haiku-20240307")
embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

Settings.llm = llm
Settings.embed_model = embed_model

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

... (2 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
from pathlib import Path

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: metagpt.pdf
Getting tools for paper: longlora.pdf
Getting tools for paper: selfrag.pdf


In [9]:
initial_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [10]:
len(initial_tools)

6

In [11]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner

agent_worker = FunctionCallingAgentWorker.from_tools(
    initial_tools,
    llm=llm,
    verbose=True
)
agent = AgentRunner(agent_worker)

In [12]:
response = agent.query(
    "Fale-me sobre o conjunto de dados de avaliação usado no LongLoRA, "
    "e depois fale-me sobre os resultados da avaliação"
)

Added user message to memory: Fale-me sobre o conjunto de dados de avaliação usado no LongLoRA, e depois fale-me sobre os resultados da avaliação
=== Calling Function ===
Calling function: vector_tool_longlora with args: {"query": "conjunto de dados de avalia\u00e7\u00e3o usado no LongLoRA"}
=== Function Output ===
O conjunto de dados de avaliação usado no LongLoRA foi o proof-pile (Azerbayev et al., 2022) e o PG19. De acordo com o contexto fornecido, o LongLoRA demonstrou resultados promissores na modelagem de linguagem de sequências longas, com reduções significativas na perplexidade ao aumentar o tamanho do contexto de treinamento e avaliação para os modelos Llama2 de 7B, 13B e 70B.
=== Calling Function ===
Calling function: vector_tool_longlora with args: {"query": "resultados da avalia\u00e7\u00e3o do LongLoRA"}
=== Function Output ===
Based on the information provided in the context, the key results of the LongLoRA evaluation are:

1. LongLoRA demonstrates strong empirical result

In [13]:
response = agent.query("Dê-me um resumo do Self-RAG e do LongLoRA")
print(str(response))

Added user message to memory: Dê-me um resumo do Self-RAG e do LongLoRA
=== Calling Function ===
Calling function: summary_tool_selfrag with args: {"input": "Self-RAG"}
=== Function Output ===
Based on the provided context, Self-RAG refers to the Selfrag sample fragmentation procedure, which is an alternative method for extracting geochronologically useful minerals from rock samples. The key points about Self-RAG are:

1. It separates minerals along their grain boundaries, allowing for the isolation of minerals without the breakage or damage that can occur during conventional crushing and milling processes. This makes it suitable for mineral separation and morphological studies.

2. The Selfrag device offers two main fragmentation options: one with a mesh sieve for grain size separation, and one without a sieve that collects all the material at the bottom of the sample container.

3. The procedure involves assembling the sample container with various components, filling it with water, 

## 2. Setup an agent over 11 papers

### Download 11 ICLR papers

In [18]:
urls = [
    "https://openreview.net/pdf?id=VtmBAGCN7o",
    "https://openreview.net/pdf?id=6PmJoRfdaK",
    "https://openreview.net/pdf?id=LzPWWPAdY4",
    "https://openreview.net/pdf?id=VTF8yNQM66",
    "https://openreview.net/pdf?id=hSyW5go0v8",
    "https://openreview.net/pdf?id=9WD9KwssyT",
    "https://openreview.net/pdf?id=yV6fD7LYkF",
    "https://openreview.net/pdf?id=hnrB5YHoYu",
    "https://openreview.net/pdf?id=WbWtOYIzIK",
    # "https://openreview.net/pdf?id=c5pwL0Soay",
    # "https://openreview.net/pdf?id=TpD2aG1h0D"
]

papers = [
    "metagpt.pdf",
    "longlora.pdf",
    "loftq.pdf",
    "swebench.pdf",
    "selfrag.pdf",
    "zipformer.pdf",
    "values.pdf",
    "finetune_fair_diffusion.pdf",
    "knowledge_card.pdf",
    # "metra.pdf",
    # "vr_mcl.pdf"
]

In [ ]:
for url, paper in zip(urls, papers):
      !wget "{url}" -O "{paper}"

In [19]:
from pathlib import Path

paper_to_tools_dict = {}
for paper in papers:
    print(f"Getting tools for paper: {paper}")
    vector_tool, summary_tool = get_doc_tools(paper, Path(paper).stem)
    paper_to_tools_dict[paper] = [vector_tool, summary_tool]

Getting tools for paper: metagpt.pdf
Getting tools for paper: longlora.pdf
Getting tools for paper: loftq.pdf
Getting tools for paper: swebench.pdf
Getting tools for paper: selfrag.pdf
Getting tools for paper: zipformer.pdf
Getting tools for paper: values.pdf
Getting tools for paper: finetune_fair_diffusion.pdf
Getting tools for paper: knowledge_card.pdf


To download these papers, below is the needed code:


    #for url, paper in zip(urls, papers):
         #!wget "{url}" -O "{paper}"
    
    
**Note**: The pdf files are included with this lesson. To access these papers, go to the `File` menu and select`Open...`.

### Estenda o agente com recuperação de ferramentas

In [20]:
all_tools = [t for paper in papers for t in paper_to_tools_dict[paper]]

In [21]:
# define an "object" index and retriever over these tools
from llama_index.core import VectorStoreIndex
from llama_index.core.objects import ObjectIndex

obj_index = ObjectIndex.from_objects(
    all_tools,
    index_cls=VectorStoreIndex,
)

In [22]:
obj_retriever = obj_index.as_retriever(similarity_top_k=3)

In [23]:
tools = obj_retriever.retrieve(
    "Fale-me sobre o conjunto de dados de avaliação usado \
     no MetaGPT e no SWE-Bench"
)

In [24]:
tools[2].metadata

ToolMetadata(description='Use ONLY IF you want to get a holistic summary of MetaGPT. Do NOT use if you have specific questions over MetaGPT.', name='summary_tool_longlora', fn_schema=<class 'llama_index.core.tools.types.DefaultToolFnSchema'>, return_direct=False)

In [25]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner

agent_worker = FunctionCallingAgentWorker.from_tools(
    tool_retriever=obj_retriever,
    llm=llm,
    system_prompt=""" \
Você é um agente criado para responder a perguntas sobre um conjunto de documentos fornecidos.
Por favor, sempre use as ferramentas fornecidas para responder a uma pergunta. Não confie em conhecimento prévio.\
""",
    verbose=True
)
agent = AgentRunner(agent_worker)

In [26]:
response = agent.query(
    "Fale-me sobre o conjunto de dados de avaliação usado \
     no MetaGPT e no SWE-Bench"
)
print(str(response))

Added user message to memory: Fale-me sobre o conjunto de dados de avaliação usado      no MetaGPT e no SWE-Bench
=== Calling Function ===
Calling function: summary_tool_finetune_fair_diffusion with args: {"input": "MetaGPT dataset"}
=== Function Output ===
I do not have any information about a "MetaGPT dataset" based on the context provided. The given text does not mention or describe anything related to a dataset called "MetaGPT". Without any additional details about this dataset, I cannot provide any factual information in response to the query.
=== Calling Function ===
Calling function: summary_tool_swebench with args: {"input": "SWE-Bench dataset"}
=== Function Output ===
SWE-Bench is a benchmark for evaluating language models on real-world software engineering tasks. It consists of 2,294 software engineering problems drawn from real GitHub issues and corresponding pull requests across 12 popular Python repositories. Given a codebase along with a description of an issue to be reso

In [27]:
response = agent.query(
  "Compare e contraste os artigos LoRA (LongLoRA, LoftQ)."
  "Analise a abordagem em cada artigo primeiro."
)

Added user message to memory: Compare and contrast the LoRA papers (LongLoRA, LoftQ). Analyze the approach in each paper first. 
=== LLM Response ===
Okay, let's analyze the approaches in the LoRA papers using the vector search tools provided.
=== Calling Function ===
Calling function: vector_tool_longlora with args: {"query": "LoRA papers approach"}
=== Function Output ===
The context information does not provide details about LoRA papers or their approach. The passage focuses on discussing the performance of the authors' LongLoRA model compared to the Llama2 7B model on a passkey retrieval task. It mentions that the authors fine-tuned the Llama2 7B model using their proposed S2-Attn attention mechanism, but does not go into specifics about the LoRA approach. Without additional context about LoRA papers and their approach, I cannot provide a meaningful answer to the query.
=== Calling Function ===
Calling function: vector_tool_loftq with args: {"query": "LoRA papers approach"}
=== Fun